In [1]:
import time
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.special import expit
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegressionCV, LassoCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, StratifiedKFold

import seaborn as sns
from econml.dr import DRLearner, LinearDRLearner, SparseLinearDRLearner, ForestDRLearner

# Importing dataset:

dataset = pd.read_stata(
    r"C:\Users\jonat\OneDrive\Desktop\Uni\Gerber et al 2008\GerberGreenLarimer_APSR_2008_social_pressure.dta",
    convert_categoricals=False
)

# Defining binary treatment and outcome variable:
dataset["D"] = (dataset["treatment"] != 0).astype(int)
dataset["Y"] = dataset["voted"]
X = ["sex", "g2000", "g2002", "p2000", "p2002", "p2004", "hh_size"]

# Selection parameters:
S = "yob"
alpha = 0.0
gamma = 1.0
seed = 1465

rng = np.random.default_rng(seed)

# Standardizing year of birth (biased variable):
S_std = (dataset[S] - dataset[S].mean()) / dataset[S].std()

# Treated = +1, Control = -1
sign = np.where(dataset["D"] == 1, 1, -1)

# Logistic selection probability:
u = alpha + gamma * sign * S_std
p_keep = expit(u)

# Constructing biased sample:
keep = rng.uniform(size=len(dataset)) < p_keep
biased = dataset.loc[keep].copy()

COVS_WITH_S = X + [S]
COVS_NO_S = X

# Settings
N_BOOT = 20
MAIN = "LinearDRLearner"
CV_FOLDS = 2
ALPHA_CI = 0.05

In [2]:
# Nuisance models (identical across all DR learners)

def outcome_model():
    """g_t(X,W) = E[Y | T, X, W]. Gradient boosted trees (EconML guide example);
    n_estimators=50, max_depth=3 are values from the guide's parameter grid."""
    return GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=seed)


def propensity_model():
    """p(X,W) = Pr[D = 1 | X, W]. Cross-validated logistic regression
    (EconML guide example). Log-odds of D in the biased sample are linear
    in S_std by construction, so a logit is correctly specified."""
    return LogisticRegressionCV()


def outcome_model_linear():
    """Linear (Lasso) outcome model, used with SparseLinearDRLearner."""
    return LassoCV()


def covariate_matrix(df, covs):
    """Standardize covariates (yob is on a 1900s scale) so the logit converges.
    Rescaling covariates does not change the ATE."""
    return StandardScaler().fit_transform(df[covs].to_numpy(dtype=float))

In [3]:
# The four EconML DR learners (Step 2)

def make_learner(kind, rs):
    if kind == "LinearDRLearner":
        return LinearDRLearner(model_regression=outcome_model(),
                               model_propensity=propensity_model(), cv=CV_FOLDS)
    if kind == "ForestDRLearner":
        return ForestDRLearner(model_regression=outcome_model(),
                               model_propensity=propensity_model(), cv=CV_FOLDS,
                               n_estimators=100, random_state=rs)
    if kind == "DRLearner":
        return DRLearner(model_regression=outcome_model(),
                         model_propensity=propensity_model(),
                         model_final=GradientBoostingRegressor(random_state=rs),
                         cv=CV_FOLDS)
    if kind == "SparseLinearDRLearner":
        return SparseLinearDRLearner(model_regression=outcome_model_linear(),
                                     model_propensity=propensity_model(), cv=CV_FOLDS)
    raise ValueError(kind)


LEARNERS = ["LinearDRLearner", "ForestDRLearner", "DRLearner", "SparseLinearDRLearner"]


def dr_ate(kind, df, covs, rs):
    """Fit one DR learner; ATE = average of est.effect(X, T0=0, T1=1)."""
    Xm = covariate_matrix(df, covs)
    est = make_learner(kind, rs)
    est.fit(df["Y"].to_numpy(), df["D"].to_numpy(), X=Xm)
    return float(np.mean(est.effect(Xm, T0=0, T1=1))), est

In [4]:
# 3. Simple estimators: naive, RCT benchmark, IPW

def diff_means(df):
    return df.loc[df.D == 1, "Y"].mean() - df.loc[df.D == 0, "Y"].mean()


def ipw_ate(df, covs):
    """Normalized (Hajek) IPW with the same logistic propensity model."""
    Xm = covariate_matrix(df, covs)
    d, y = df["D"].to_numpy(), df["Y"].to_numpy()
    ps = propensity_model().fit(Xm, d).predict_proba(Xm)[:, 1]
    w1, w0 = d / ps, (1 - d) / (1 - ps)
    return np.sum(w1 * y) / np.sum(w1) - np.sum(w0 * y) / np.sum(w0)

In [5]:
# Nonparametric bootstrap (resample rows, refit everything)

def bootstrap(stat_fn, df, B=N_BOOT, label="", rs=seed):
    brng = np.random.default_rng(rs)
    n = len(df)
    draws = []
    t0 = time.time()
    for b in range(B):
        idx = brng.integers(0, n, n)
        draws.append(stat_fn(df.iloc[idx].reset_index(drop=True), b))
        if (b + 1) % 10 == 0:
            print(f"   [{label}] {b + 1}/{B} replicates, {time.time() - t0:.0f}s")
    draws = np.array(draws)
    lo, hi = np.percentile(draws, [100 * ALPHA_CI / 2, 100 * (1 - ALPHA_CI / 2)])
    return lo, hi, draws.std(ddof=1)


results = []


def record(name, sample, covs, est, lo, hi, se):
    results.append(dict(Estimator=name, Sample=sample, Covariates=covs,
                        ATE=est, CI_low=lo, CI_high=hi, Boot_SE=se))
    print(f"{name:<45s} {covs:<10s} ATE={est: .4f}  95% CI [{lo: .4f}, {hi: .4f}]")


print(f"Full sample n = {len(dataset):,} | Biased sample n = {len(biased):,} "
      f"| share treated in biased sample = {biased.D.mean():.3f}")

# -- IPW (Week 7 comparison; replace with your Week 7 number if it differs) ---
tau_ipw = ipw_ate(biased, COVS_WITH_S)
lo, hi, se = bootstrap(lambda d, b: ipw_ate(d, COVS_WITH_S), biased, label="IPW")
record("IPW (Hajek, logit PS)", "biased", "with S", tau_ipw, lo, hi, se)

Full sample n = 344,084 | Biased sample n = 171,990 | share treated in biased sample = 0.443
   [IPW] 10/20 replicates, 13s
   [IPW] 20/20 replicates, 25s
IPW (Hajek, logit PS)                         with S     ATE= 0.0435  95% CI [ 0.0372,  0.0487]


In [ ]:
# STEP 1 + 2b: DR learners WITH S in the covariate set
# STEP 3     : the same DR learners WITHOUT S

fitted = {}
for covs, label in [(COVS_WITH_S, "with S"), (COVS_NO_S, "without S")]:
    print(f"\n=== DR learners, covariates {label} ===")
    for kind in LEARNERS:
        point, est = dr_ate(kind, biased, covs, seed)
        fitted[(kind, label)] = est
        if kind == MAIN:
            lo, hi, se = bootstrap(lambda d, b, k=kind, c=covs: dr_ate(k, d, c, seed + b)[0],
                                   biased, label=f"{kind}, {label}")
        else:
            lo, hi, se = np.nan, np.nan, np.nan
        record(kind, "biased", label, point, lo, hi, se)

# Cross-check: LinearDRLearner with all covariates as controls (W) and no X gives a
# constant effect = ATE, with an analytic CI from est.effect_interval (EconML guide)
Wm = covariate_matrix(biased, COVS_WITH_S)
lin = LinearDRLearner(model_regression=outcome_model(), model_propensity=propensity_model(),
                      cv=CV_FOLDS)
lin.fit(biased["Y"].to_numpy(), biased["D"].to_numpy(), X=None, W=Wm)
print("\nCross-check, LinearDRLearner analytic ATE and CI (with S):",
      np.round(lin.effect(T0=0, T1=1), 4),
      np.round(np.ravel(lin.effect_interval(T0=0, T1=1, alpha=ALPHA_CI)), 4))

# Final-stage score (EconML guide: est.score_, lower is better)
for kind in LEARNERS:
    print(f"score_ {kind:<25s} with S: {fitted[(kind, 'with S')].score_:.5f}")


=== DR learners, covariates with S ===
   [LinearDRLearner, with S] 10/20 replicates, 162s


In [8]:
# PART 2

tau_rct = diff_means(dataset)

tau_dr_full, est_full = dr_ate(MAIN, dataset, COVS_WITH_S, seed)

# Optional: formula-based CI, same approach as the earlier cross-check (no bootstrap)
W_full = covariate_matrix(dataset, COVS_WITH_S)
lin_full = LinearDRLearner(model_regression=outcome_model(),
                           model_propensity=propensity_model(), cv=CV_FOLDS)
lin_full.fit(dataset["Y"].to_numpy(), dataset["D"].to_numpy(), X=None, W=W_full)
lo_full, hi_full = np.ravel(lin_full.effect_interval(T0=0, T1=1, alpha=ALPHA_CI))

print(f"\nDR on full sample ({MAIN}, with S): ATE = {tau_dr_full:.4f} "
      f"(95% CI [{lo_full:.4f}, {hi_full:.4f}]) vs tau_RCT = {tau_rct:.4f} "
      f"(difference {tau_dr_full - tau_rct:+.4f})")


DR on full sample (LinearDRLearner, with S): ATE = 0.0427 (95% CI [0.0397, 0.0457]) vs tau_RCT = 0.0434 (difference -0.0006)
